In [ ]:
#| default_exp plots

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
from typing import Any

In [ ]:
#| export
import holoviews as hv
import hvplot.pandas  # noqa: F401 - registers the pandas .hvplot accessor
import pandas as pd
from bokeh.models.formatters import NumeralTickFormatter

In [ ]:
#| export
DATE_COL = "date"
SERIES_COL = "series"
VALUE_COL = "value"

In [ ]:
#| export
def _to_long_timeseries(
    data: pd.Series | pd.DataFrame,
    *,
    date_col: str = DATE_COL,
    series_col: str = SERIES_COL,
    value_col: str = VALUE_COL,
    default_series_name: str = "value",
) -> pd.DataFrame:
    """Convert a Series or wide DataFrame to a stable long-form schema."""
    if isinstance(data, pd.Series):
        series_name = str(data.name) if data.name is not None else default_series_name
        return (
            data.rename(value_col)
            .rename_axis(date_col)
            .reset_index()
            .assign(**{series_col: series_name})
            [[date_col, series_col, value_col]]
        )

    if isinstance(data, pd.DataFrame):
        return (
            data.rename_axis(index=date_col, columns=series_col)
            .reset_index()
            .melt(id_vars=date_col, var_name=series_col, value_name=value_col)
            [[date_col, series_col, value_col]]
        )

    msg = "data must be a pandas Series or DataFrame"
    raise TypeError(msg)

In [ ]:
#| export
def _value_formats(
    *,
    is_perc: bool,
    value_format: str | None,
    axis_format: str | None,
) -> tuple[str, NumeralTickFormatter | None]:
    hover_format = value_format or ("0.00%" if is_perc else "0.00")

    if axis_format is None and not is_perc:
        return hover_format, NumeralTickFormatter(format="0.00")

    formatter_format = axis_format if axis_format is not None else "0.0%"
    return hover_format, NumeralTickFormatter(format=formatter_format)

In [ ]:
#| export
def timeseries_plot(
    data: pd.Series | pd.DataFrame,
    *,
    is_perc: bool = True,
    title: str | None = None,
    value_label: str | None = None,
    series_label: str = "Series",
    date_label: str = "Date",
    date_format: str = "%b %Y",
    value_format: str | None = None,
    axis_format: str | None = None,
    show_series_in_hover: bool | None = None,
    legend: str | bool = "top_left",
    hline: float | None = None,
    hline_opts: dict[str, Any] | None = None,
    height: int = 400,
    max_width: int = 1000,
    responsive: bool = True,
    shared_axes: bool = False,
    **kwargs: Any,
) -> Any:
    """Plot a pandas time series with consistent Bokeh hover tooltips.

    The input may be a single Series or a wide DataFrame. Both are normalized
    to ``date``, ``series``, and ``value`` columns so one tooltip definition
    works for all supported inputs.
    """
    plot_data = _to_long_timeseries(data)
    unique_series = plot_data[SERIES_COL].nunique(dropna=False)
    include_series = show_series_in_hover
    if include_series is None:
        include_series = unique_series > 1

    hover_value_format, yformatter = _value_formats(
        is_perc=is_perc,
        value_format=value_format,
        axis_format=axis_format,
    )
    value_label = value_label or ("Return" if is_perc else "Value")

    hover_tooltips = [
        (date_label, f"@{{{DATE_COL}}}{{{date_format}}}"),
        (value_label, f"@{{{VALUE_COL}}}{{{hover_value_format}}}"),
    ]
    if include_series:
        hover_tooltips.insert(1, (series_label, f"@{{{SERIES_COL}}}"))

    plot = plot_data.hvplot.line(
        x=DATE_COL,
        y=VALUE_COL,
        by=SERIES_COL,
        title=title,
        height=height,
        responsive=responsive,
        max_width=max_width,
        padding=0.01,
        autorange="y",
        hover="vline",
        hover_tooltips=hover_tooltips,
        yformatter=yformatter,
        toolbar="above",
        legend=legend,
        **kwargs,
    ).opts(shared_axes=shared_axes)

    if hline is None:
        return plot

    line_opts = {"color": "gray", "line_dash": "dashed", "line_width": 1}
    line_opts.update(hline_opts or {})
    return (plot * hv.HLine(hline).opts(**line_opts)).opts(
        height=height,
        responsive=responsive,
        max_width=max_width,
        shared_axes=shared_axes,
    )